[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/lab1.ipynb)

# Lab 1: The Gene That Wasn't There
## The False Discovery Problem

**Day 1, 10:00–11:00** | Learning Outcomes: LO1, LO3

In the lecture, you heard the story: a serotonin gene that wasn't there, a dead salmon with brain activity, and a field that spent fifteen years chasing false positives. Now you're going to see it happen — in your own code, on real data.

By the end of this lab you will:
1. Run 3,051 hypothesis tests on the Golub leukemia dataset
2. Read a p-value histogram and know what it tells you
3. Feel the gap between Bonferroni and no correction
4. Use AI to extend the analysis, then evaluate its output

The Golub dataset stays with us for the entire workshop. This is the first time you'll touch it; by the end of Day 2 you'll have tested it, reduced it, and built predictive models from it.

In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

---
## Part 1 — Meet the Data

The **Golub** dataset is a landmark microarray study from Golub et al. (1999). It measures expression levels of **3,051 genes** across **72 patients** diagnosed with either **ALL** (acute lymphoblastic leukemia) or **AML** (acute myeloid leukemia).

This is a classic high-dimensional setting: $p = 3{,}051$ features and $n = 72$ observations. The ratio $p/n \approx 42$ means we have roughly 42 times more variables than samples. Every method we study in this workshop exists because of this imbalance.

In [ ]:
# Load the Golub 1999 leukemia dataset from Zenodo
!pip install -q pyreadr
import pyreadr, urllib.request, os

url = "https://zenodo.org/records/8123245/files/leukemia_data_Golub99_3051.rda?download=1"
if not os.path.exists("golub.rda"):
    urllib.request.urlretrieve(url, "golub.rda")

rda = pyreadr.read_r("golub.rda")
expr = pd.concat([rda['golub_train_3051'], rda['golub_test_3051']]).reset_index(drop=True)
labels = pd.concat([rda['golub_train_response'].iloc[:, 0],
                     rda['golub_test_response'].iloc[:, 0]]).reset_index(drop=True)

# expr: 72 samples (rows) x 3,051 genes (columns)
# labels: "ALL" or "AML" for each sample

print(f"Expression matrix shape: {expr.shape}")
print(f"Samples (n): {expr.shape[0]}")
print(f"Genes (p): {expr.shape[1]}")

label_counts = labels.value_counts()
print(f"ALL samples: {label_counts.get('ALL', 0)}")
print(f"AML samples: {label_counts.get('AML', 0)}")
print(f"Ratio p/n: {expr.shape[1] / expr.shape[0]:.1f}")

# Create sample masks for the two groups
all_mask = labels == 'ALL'
aml_mask = labels == 'AML'

print(f"\nFirst few genes: {expr.columns[:5].tolist()}")

### Exercise 1.1

Look at the ratio $p/n$. You have roughly 42 times more variables than observations. If you tried ordinary linear regression with all 3,051 genes as predictors, what would happen? (We'll return to this on Day 2.)

**YOUR ANSWER:**



---
## Part 2 — One Gene, One Test

Before we test thousands of genes at once, let's start with familiar territory: one gene, one hypothesis test. We pick a single gene, compare its expression between ALL and AML patients, and ask whether the difference is statistically significant.

In [ ]:
# Pick the first gene
gene_name = expr.columns[0]
gene_expr = expr[gene_name]

all_expr = gene_expr[all_mask].values
aml_expr = gene_expr[aml_mask].values

# Boxplot comparing ALL vs AML expression
fig, ax = plt.subplots(figsize=(6, 5))
bp = ax.boxplot([all_expr, aml_expr], labels=['ALL', 'AML'],
                patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#4C72B0')
bp['boxes'][1].set_facecolor('#DD8452')
ax.set_ylabel('Expression level')
ax.set_title(f'Gene: {gene_name}')

# Two-sample t-test (unequal variance)
t_stat, p_val = stats.ttest_ind(all_expr, aml_expr, equal_var=False)
ax.text(0.95, 0.95, f't = {t_stat:.3f}\np = {p_val:.4f}',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

print(f"Gene: {gene_name}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.4f}")

One gene, one test, one p-value. If the p-value is below 0.05, you'd call the gene significant. But you don't have one gene. You have 3,051.

---
## Part 3 — 3,051 Tests

Now we do what a real genomics study does: test every gene for differential expression between ALL and AML leukemia.

In [ ]:
# Run a t-test for every gene
gene_names = expr.columns.tolist()
n_genes = len(gene_names)
pvals = np.zeros(n_genes)

for i, gene in enumerate(gene_names):
    all_vals = expr.loc[all_mask, gene].values.astype(float)
    aml_vals = expr.loc[aml_mask, gene].values.astype(float)
    _, pvals[i] = stats.ttest_ind(all_vals, aml_vals, equal_var=False)

alpha = 0.05
n_significant = np.sum(pvals < alpha)
expected_fp = n_genes * alpha

print(f"Total genes tested: {n_genes}")
print(f"Significant at \u03b1 = 0.05 (no correction): {n_significant}")
print(f"Expected false positives if nothing were real: {expected_fp:.0f}")

### Exercise 3.1

How many genes were called significant? How many false positives would you *expect* if none of the genes were truly differentially expressed? Where does that expected number come from? (Hint: $3{,}051 \times 0.05 = ?$)

**YOUR ANSWER:**



---
## Part 4 — The P-Value Histogram

This is the most useful plot in high-dimensional statistics. Before you run *any* correction method, plot the histogram of your p-values. It tells you three things at a glance:
1. Whether there is any signal at all (spike near zero)
2. Roughly what fraction of tests are true nulls ($\pi_0$, estimated from the flat part)
3. Whether something went wrong with your analysis (strange shapes)

In [ ]:
# P-value histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pvals, bins=100, edgecolor='white', color='#4C72B0', alpha=0.8)

# Expected height under uniform null: n_genes / n_bins
expected_height = n_genes / 100
ax.axhline(y=expected_height, color='red', linestyle='--', linewidth=2,
           label=f'Expected under null ({expected_height:.0f} per bin)')

ax.set_xlabel('P-value')
ax.set_ylabel('Frequency')
ax.set_title('P-value histogram: 3,051 gene-level t-tests')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 4.1

Describe what you see. Where is the signal? Where is the noise?

**YOUR ANSWER:**

### Exercise 4.2

Estimate $\pi_0$ (the proportion of true nulls) by eye. Look at the flat region of the histogram and compare its height to the total number of genes.

**YOUR ANSWER:**



### Verify with a pure-null simulation

To understand what "no signal" looks like, we'll simulate 3,051 pure-null t-tests (both groups drawn from the same distribution) and compare.

In [ ]:
# Null simulation: 3,051 tests where both groups come from N(0,1)
# Use the actual group sizes from the Golub dataset (47 ALL, 25 AML)
np.random.seed(2026)
n_all = int(all_mask.sum())   # 47
n_aml = int(aml_mask.sum())   # 25
null_pvals = np.zeros(n_genes)

for i in range(n_genes):
    x = np.random.normal(0, 1, n_all)
    y = np.random.normal(0, 1, n_aml)
    _, null_pvals[i] = stats.ttest_ind(x, y, equal_var=False)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(null_pvals, bins=100, edgecolor='white', color='#999999', alpha=0.8)
axes[0].axhline(y=expected_height, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('P-value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Simulated null (no real signal)')

axes[1].hist(pvals, bins=100, edgecolor='white', color='#4C72B0', alpha=0.8)
axes[1].axhline(y=expected_height, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('P-value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Real data (Golub dataset)')

plt.tight_layout()
plt.show()

### Exercise 4.3

How many "significant" genes appear in the null simulation? Is it close to 5% of 3,051?

**YOUR ANSWER:**



In [ ]:
# Count false positives in the null simulation
null_sig = np.sum(null_pvals < 0.05)
print(f"False positives in null simulation: {null_sig}")
print(f"Expected (5% of {n_genes}): {n_genes * 0.05:.0f}")
print(f"Actual percentage: {100 * null_sig / n_genes:.1f}%")

---
## Part 5 — Bonferroni vs. Benjamini-Hochberg

We have thousands of significant p-values, but we know many could be false positives. Two classic corrections control different error rates:

- **Bonferroni** controls the family-wise error rate (FWER) — the probability of *any* false positive. Divide $\alpha$ by the number of tests.
- **Benjamini-Hochberg** controls the false discovery rate (FDR) — the expected fraction of false positives among declared discoveries. Allows more discoveries by tolerating a controlled fraction of false positives.

In [ ]:
# Bonferroni correction
bonf_threshold = 0.05 / n_genes
n_bonf = np.sum(pvals < bonf_threshold)

print(f"Bonferroni threshold: {bonf_threshold:.2e}")
print(f"Genes surviving Bonferroni: {n_bonf}")

In [ ]:
# Benjamini-Hochberg correction
reject_bh_05, pvals_bh_05, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
reject_bh_10, pvals_bh_10, _, _ = multipletests(pvals, alpha=0.10, method='fdr_bh')

n_bh_05 = np.sum(reject_bh_05)
n_bh_10 = np.sum(reject_bh_10)

print(f"Genes significant at FDR \u2264 0.05 (BH): {n_bh_05}")
print(f"Genes significant at FDR \u2264 0.10 (BH): {n_bh_10}")

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Method': ['No correction (α = 0.05)', 'Bonferroni (α = 0.05)',
               'BH (FDR ≤ 0.05)', 'BH (FDR ≤ 0.10)'],
    'Genes called significant': [np.sum(pvals < 0.05), n_bonf, n_bh_05, n_bh_10]
})
print(comparison.to_string(index=False))

### Exercise 5.1

You're a biologist with funding to follow up 20 genes with qPCR validation. Which gene list would you use and why?

**YOUR ANSWER:**

### Exercise 5.2

A collaborator says "Just use Bonferroni — it's the safest." Write 2–3 sentences explaining why that might not be the best choice here.

**YOUR ANSWER:**



---
## Part 6 — What Goes Wrong Without Correction

The multiple testing problem doesn't just affect individual studies. When combined with publication bias and the winner's curse, it creates systematic distortions in the scientific literature. Let's simulate both.

In [ ]:
# Publication bias simulation
# 1,000 independent labs each test ONE null gene (no real effect)
np.random.seed(2026)
n_labs = 1000
n_per_group = 30
lab_pvals = np.zeros(n_labs)

for i in range(n_labs):
    x = np.random.normal(0, 1, n_per_group)
    y = np.random.normal(0, 1, n_per_group)  # same distribution — no real effect
    _, lab_pvals[i] = stats.ttest_ind(x, y, equal_var=False)

# Only "significant" results get published
published = lab_pvals < 0.05
n_published = np.sum(published)

print(f"Labs that ran the study: {n_labs}")
print(f"Labs that found p < 0.05 (published): {n_published}")
print(f"Fraction published: {n_published / n_labs:.1%}")
print()
print(f"In the published literature:")
print(f"  Studies showing 'significant' effect: {n_published}/{n_published} = 100%")
print(f"  A reader sees only published papers and concludes the effect is real.")
print(f"  But the true effect is ZERO.")

### Exercise 6.1

What fraction of all studies found a significant effect? What fraction of *published* studies show a significant effect? This gap is publication bias. Explain in your own words why this is dangerous.

**YOUR ANSWER:**



In [ ]:
# Winner's curse simulation
# 1,000 studies with a REAL but small effect (d = 0.3)
np.random.seed(2026)
true_effect = 0.3
n_studies = 1000
n_per_group = 30
observed_effects = np.zeros(n_studies)
study_pvals = np.zeros(n_studies)

for i in range(n_studies):
    x = np.random.normal(0, 1, n_per_group)
    y = np.random.normal(true_effect, 1, n_per_group)
    observed_effects[i] = np.mean(y) - np.mean(x)
    _, study_pvals[i] = stats.ttest_ind(x, y)

# Only significant studies get published
published_mask = study_pvals < 0.05
published_effects = observed_effects[published_mask]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(published_effects, bins=30, edgecolor='white', color='#DD8452', alpha=0.8,
        label=f'Published effect sizes (n={len(published_effects)})')
ax.axvline(x=true_effect, color='green', linewidth=2, linestyle='-',
           label=f'True effect (d = {true_effect})')
ax.axvline(x=np.mean(published_effects), color='red', linewidth=2, linestyle='--',
           label=f'Mean published effect (d = {np.mean(published_effects):.2f})')
ax.set_xlabel('Observed effect size (mean difference)')
ax.set_ylabel('Frequency')
ax.set_title("Winner's Curse: Published studies overestimate the true effect")
ax.legend()
plt.tight_layout()
plt.show()

print(f"True effect size: {true_effect}")
print(f"Mean effect in ALL studies: {np.mean(observed_effects):.3f}")
print(f"Mean effect in PUBLISHED studies: {np.mean(published_effects):.3f}")
print(f"Overestimation factor: {np.mean(published_effects) / true_effect:.1f}x")

### Exercise 6.2

Why is the average published effect size larger than the true effect? What mechanism causes this?

**YOUR ANSWER:**



---
## Part 7 — Extend with AI

### 7.1 Power curve

Use AI (or write code yourself) to answer: **how does BH power depend on effect size?**

Suggested prompt to give your AI assistant:

> "Using Python, simulate 10,000 genes where 200 are truly alternative with effect size d. Vary d from 0.5 to 3.0 in steps of 0.25. For each d, apply the BH procedure at FDR = 0.10 and compute power (true positives / 200). Plot power vs. effect size. Use numpy.random.seed(2026) and n = 25 per group."

In [ ]:
# YOUR CODE HERE (AI-generated or your own)


### Exercise 7.1

At what effect size does BH achieve 80% power?

**YOUR ANSWER:**



### 7.2 Critique checklist

If you used AI to generate the power curve code, evaluate it against this checklist:

- [ ] Did it set a random seed?
- [ ] Does the simulation match the setup (10,000 genes, 200 alternative)?
- [ ] Did it apply BH correction correctly (not Bonferroni, not uncorrected)?
- [ ] Does the plot have axis labels, a title, and a clear trend?
- [ ] Does power increase with effect size and approach 1.0?
- [ ] Are there any subtle errors (e.g., wrong group sizes, wrong FDR threshold)?

---
## Wrap-Up Questions

1. What does the p-value histogram tell you *before* you run any correction? (Three things.)
2. Why is the uncorrected approach dangerous when testing thousands of genes?
3. When would Bonferroni be the right choice over BH?
4. How does publication bias turn individual false positives into a field-wide problem?
5. Looking at the Golub p-value histogram: what questions does it raise that we haven't answered yet?

---
## Looking Ahead

The histogram told us there's signal in the Golub data, and Bonferroni is too blunt to find most of it. In **Lecture 2**, you'll learn how the BH procedure actually works — and what happens when Brad Efron notices that the histogram is "too wide." In **Lab 2**, you'll apply FDR methods, q-values, and the two-groups model to this same dataset.